# Initial Data Exploration for ArcticShadowTracker

This notebook provides initial exploration of Arctic maritime surveillance data including:
- Sentinel-1 SAR imagery analysis
- AIS data exploration
- Infrastructure mapping
- Basic vessel detection validation

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from datetime import datetime, timedelta

# ArcticShadowTracker modules
from detection.dark_vessels import DarkVesselDetector
from detection.cable_monitor import CableMonitor
from analysis.patterns import BehaviorPatternAnalyzer

# Set up plotting
plt.style.use('default')
sns.set_palette('viridis')
%matplotlib inline

## 1. Arctic Region Overview

Let's start by exploring the Arctic maritime environment and key areas of interest.

In [ ]:
# Define Arctic monitoring regions
arctic_regions = {
    'Svalbard': {'center': (78.22, 15.63), 'radius_km': 100},
    'Kola Peninsula': {'center': (69.07, 33.42), 'radius_km': 200},
    'Barents Sea': {'center': (74.5, 35.0), 'radius_km': 300},
    'Franz Josef Land': {'center': (80.6, 55.0), 'radius_km': 150}
}

# Create interactive map
m = folium.Map(
    location=[75.0, 30.0],
    zoom_start=4,
    tiles='OpenStreetMap'
)

# Add regions to map
for region_name, region_data in arctic_regions.items():
    folium.Circle(
        location=region_data['center'],
        radius=region_data['radius_km'] * 1000,  # Convert to meters
        popup=f"{region_name} - Monitoring Zone",
        color='blue',
        fillColor='lightblue',
        fillOpacity=0.3
    ).add_to(m)
    
    folium.Marker(
        location=region_data['center'],
        popup=region_name,
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m)

print("Arctic Monitoring Regions Map:")
m

## 2. Submarine Cable Infrastructure

Critical undersea infrastructure that requires monitoring.

In [ ]:
# Initialize cable monitor to get infrastructure data
cable_monitor = CableMonitor()

# Display cable information
print("Arctic Submarine Cables:")
print("=" * 50)

for cable in cable_monitor.cables:
    print(f"\nCable: {cable['name']}")
    print(f"Type: {cable['type']}")
    print(f"Capacity: {cable.get('capacity', 'N/A')}")
    print(f"Year Installed: {cable.get('year_installed', 'N/A')}")
    print(f"Route Points: {len(cable['route'])}")
    print(f"Depth Range: {cable.get('depth_range', 'N/A')} meters")
    
    if cable.get('status') == 'planned':
        print("Status: PLANNED - Future deployment")
    else:
        print("Status: OPERATIONAL")

In [ ]:
# Create cable infrastructure map
cable_map = folium.Map(
    location=[75.0, 20.0],
    zoom_start=5,
    tiles='OpenStreetMap'
)

# Add cables to map
colors = ['red', 'blue', 'green', 'purple', 'orange']

for i, cable in enumerate(cable_monitor.cables):
    color = colors[i % len(colors)]
    
    # Draw cable route
    folium.PolyLine(
        locations=cable['route'],
        color=color,
        weight=3,
        opacity=0.8,
        popup=f"{cable['name']} - {cable['type']}"
    ).add_to(cable_map)
    
    # Mark landing points
    for j, point in enumerate(cable['route']):
        if j == 0 or j == len(cable['route']) - 1:  # Landing points
            folium.Marker(
                location=point,
                popup=f"{cable['name']} - Landing Point",
                icon=folium.Icon(color='red', icon='anchor')
            ).add_to(cable_map)
    
    # Add protection zones
    for point in cable['route']:
        folium.Circle(
            location=point,
            radius=5000,  # 5km protection zone
            color=color,
            fillColor=color,
            fillOpacity=0.1,
            popup=f"{cable['name']} - Protection Zone"
        ).add_to(cable_map)

print("Submarine Cable Infrastructure Map:")
cable_map

## 3. Simulated Data Exploration

Let's create and explore some simulated maritime data to demonstrate the system.

In [ ]:
# Generate simulated SAR detections
np.random.seed(42)

def generate_simulated_sar_detections(n_vessels=20):
    """Generate simulated SAR vessel detections in Arctic waters."""
    detections = []
    
    for i in range(n_vessels):
        # Random position in Arctic waters
        lat = np.random.uniform(68.0, 82.0)
        lon = np.random.uniform(10.0, 60.0)
        
        detection = {
            'detection_id': f'SAR_{datetime.now().strftime("%Y%m%d")}_{i:03d}',
            'latitude': lat,
            'longitude': lon,
            'estimated_length': np.random.uniform(20, 200),
            'estimated_width': np.random.uniform(5, 30),
            'vessel_area_pixels': np.random.uniform(50, 500),
            'intensity_mean': np.random.uniform(100, 255),
            'intensity_max': np.random.uniform(150, 255),
            'confidence': np.random.uniform(0.5, 1.0),
            'detection_time': (datetime.now() - timedelta(hours=np.random.randint(0, 24))).isoformat()
        }
        
        detections.append(detection)
    
    return detections

# Generate simulated AIS data
def generate_simulated_ais_data(n_vessels=15):
    """Generate simulated AIS data."""
    ais_data = []
    
    for i in range(n_vessels):
        # Random position in Arctic waters
        lat = np.random.uniform(68.0, 82.0)
        lon = np.random.uniform(10.0, 60.0)
        
        message = {
            'mmsi': f'2{i:08d}',
            'vessel_name': f'VESSEL_{i:03d}',
            'latitude': lat,
            'longitude': lon,
            'speed_over_ground': np.random.uniform(0, 25),
            'course_over_ground': np.random.uniform(0, 360),
            'heading': np.random.uniform(0, 360),
            'ship_type': np.random.choice([30, 31, 70, 71, 80, 81]),  # Various ship types
            'length': np.random.uniform(20, 300),
            'width': np.random.uniform(5, 40),
            'timestamp': (datetime.now() - timedelta(hours=np.random.randint(0, 24))).isoformat()
        }
        
        ais_data.append(message)
    
    return ais_data

# Generate data
sar_detections = generate_simulated_sar_detections(20)
ais_data = generate_simulated_ais_data(15)

print(f"Generated {len(sar_detections)} SAR detections")
print(f"Generated {len(ais_data)} AIS messages")
print(f"Potential dark vessels: {len(sar_detections) - len(ais_data)} (if no matches)")

In [ ]:
# Analyze SAR detection characteristics
sar_df = pd.DataFrame(sar_detections)
ais_df = pd.DataFrame(ais_data)

# Plot vessel size distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# SAR vessel sizes
axes[0, 0].hist(sar_df['estimated_length'], bins=15, alpha=0.7, color='blue')
axes[0, 0].set_title('SAR Detected Vessel Lengths')
axes[0, 0].set_xlabel('Length (meters)')
axes[0, 0].set_ylabel('Count')

# SAR confidence scores
axes[0, 1].hist(sar_df['confidence'], bins=15, alpha=0.7, color='green')
axes[0, 1].set_title('SAR Detection Confidence')
axes[0, 1].set_xlabel('Confidence Score')
axes[0, 1].set_ylabel('Count')

# AIS vessel speeds
axes[1, 0].hist(ais_df['speed_over_ground'], bins=15, alpha=0.7, color='red')
axes[1, 0].set_title('AIS Reported Speeds')
axes[1, 0].set_xlabel('Speed (knots)')
axes[1, 0].set_ylabel('Count')

# Geographic distribution
axes[1, 1].scatter(sar_df['longitude'], sar_df['latitude'], alpha=0.6, color='blue', label='SAR')
axes[1, 1].scatter(ais_df['longitude'], ais_df['latitude'], alpha=0.6, color='red', label='AIS')
axes[1, 1].set_title('Vessel Geographic Distribution')
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Dark Vessel Detection

Now let's test our dark vessel detection algorithm.

In [ ]:
# Initialize dark vessel detector
detector = DarkVesselDetector(matching_threshold_meters=500)

# Find dark vessels
dark_vessels = detector.find_dark_vessels(sar_detections, ais_data)

print(f"\nDark Vessel Detection Results:")
print(f"="*40)
print(f"SAR detections: {len(sar_detections)}")
print(f"AIS messages: {len(ais_data)}")
print(f"Dark vessels found: {len(dark_vessels)}")
print(f"Detection rate: {len(dark_vessels)/len(sar_detections)*100:.1f}%")

if dark_vessels:
    print(f"\nDark Vessel Details:")
    for i, vessel in enumerate(dark_vessels[:5]):  # Show first 5
        print(f"\nVessel {i+1}:")
        print(f"  ID: {vessel['detection_id']}")
        print(f"  Position: {vessel['latitude']:.3f}, {vessel['longitude']:.3f}")
        print(f"  Size: {vessel['estimated_length']:.1f}m x {vessel['estimated_width']:.1f}m")
        print(f"  Confidence: {vessel['confidence']:.2f}")
        print(f"  Risk Score: {vessel['risk_score']:.1f}/10")

In [ ]:
# Create detection visualization map
detection_map = folium.Map(
    location=[75.0, 30.0],
    zoom_start=4,
    tiles='OpenStreetMap'
)

# Add SAR detections (all vessels)
for detection in sar_detections:
    folium.CircleMarker(
        location=[detection['latitude'], detection['longitude']],
        radius=5,
        popup=f"SAR Detection: {detection['detection_id']}",
        color='blue',
        fillColor='lightblue',
        fillOpacity=0.7
    ).add_to(detection_map)

# Add AIS vessels
for ais in ais_data:
    folium.CircleMarker(
        location=[ais['latitude'], ais['longitude']],
        radius=5,
        popup=f"AIS Vessel: {ais['vessel_name']}",
        color='green',
        fillColor='lightgreen',
        fillOpacity=0.7
    ).add_to(detection_map)

# Highlight dark vessels
for vessel in dark_vessels:
    folium.CircleMarker(
        location=[vessel['latitude'], vessel['longitude']],
        radius=8,
        popup=f"DARK VESSEL: {vessel['detection_id']}<br>Risk: {vessel['risk_score']:.1f}/10",
        color='red',
        fillColor='red',
        fillOpacity=0.8
    ).add_to(detection_map)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 150px; height: 90px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<b>Legend</b><br>
<i class="fa fa-circle" style="color:blue"></i> SAR Detection<br>
<i class="fa fa-circle" style="color:green"></i> AIS Vessel<br>
<i class="fa fa-circle" style="color:red"></i> Dark Vessel
</div>
'''
detection_map.get_root().html.add_child(folium.Element(legend_html))

print("Dark Vessel Detection Map:")
detection_map

## 5. Cable Proximity Analysis

Analyze which vessels are near critical submarine cables.

In [ ]:
# Check all vessels for cable proximity
all_vessels = dark_vessels + [
    {
        'vessel_id': ais['mmsi'],
        'latitude': ais['latitude'],
        'longitude': ais['longitude'],
        'vessel_name': ais['vessel_name'],
        'source': 'AIS'
    } for ais in ais_data
]

# Add source info to dark vessels
for vessel in dark_vessels:
    vessel['source'] = 'Dark Vessel'

# Check cable proximity
vessels_with_cable_info = cable_monitor.check_vessel_cable_proximity(all_vessels)

# Find vessels near cables
vessels_near_cables = [v for v in vessels_with_cable_info if v.get('near_cable', False)]

print(f"\nCable Proximity Analysis:")
print(f"="*40)
print(f"Total vessels analyzed: {len(all_vessels)}")
print(f"Vessels near cables: {len(vessels_near_cables)}")

if vessels_near_cables:
    print(f"\nVessels Near Critical Infrastructure:")
    for vessel in vessels_near_cables:
        print(f"\nVessel: {vessel.get('vessel_name', vessel['vessel_id'])}")
        print(f"Source: {vessel['source']}")
        print(f"Closest Cable: {vessel['closest_cable']}")
        print(f"Distance: {vessel['distance_to_cable_km']:.1f} km")
        
        for alert in vessel.get('cable_alerts', []):
            print(f"  ALERT: {alert['alert_level']} - {alert['cable_name']} ({alert['distance_km']:.1f}km)")

## 6. Risk Assessment Summary

Generate overall risk assessment for the monitoring area.

In [ ]:
# Generate comprehensive detection report
detection_report = detector.generate_detection_report(dark_vessels)

print("\n" + "="*60)
print("ARCTIC SHADOW TRACKER - DETECTION REPORT")
print("="*60)

print(f"\nReport Timestamp: {detection_report['summary']['analysis_time']}")
print(f"\nSUMMARY:")
print(f"  Total Dark Vessels: {detection_report['summary']['total_dark_vessels']}")
print(f"  High Risk Vessels: {detection_report['summary']['high_risk_vessels']}")

if detection_report['summary']['total_dark_vessels'] > 0:
    avg_risk = np.mean([v.get('risk_score', 0) for v in dark_vessels])
    print(f"  Average Risk Score: {avg_risk:.1f}/10")
    
    # Geographic distribution
    bounds = detection_report['summary'].get('geographic_bounds', {})
    if bounds:
        print(f"\nGEOGRAPHIC COVERAGE:")
        print(f"  Northern extent: {bounds['north']:.2f}°N")
        print(f"  Southern extent: {bounds['south']:.2f}°N")
        print(f"  Eastern extent: {bounds['east']:.2f}°E")
        print(f"  Western extent: {bounds['west']:.2f}°E")

# Risk level distribution
risk_levels = {'LOW': 0, 'MEDIUM': 0, 'HIGH': 0, 'CRITICAL': 0}
for vessel in dark_vessels:
    risk_score = vessel.get('risk_score', 0)
    if risk_score >= 8:
        risk_levels['CRITICAL'] += 1
    elif risk_score >= 6:
        risk_levels['HIGH'] += 1
    elif risk_score >= 4:
        risk_levels['MEDIUM'] += 1
    else:
        risk_levels['LOW'] += 1

print(f"\nRISK LEVEL DISTRIBUTION:")
for level, count in risk_levels.items():
    if count > 0:
        print(f"  {level}: {count} vessels")

print(f"\nINFRASTRUCTURE THREATS:")
if vessels_near_cables:
    cable_threat_count = len(vessels_near_cables)
    dark_near_cables = len([v for v in vessels_near_cables if v['source'] == 'Dark Vessel'])
    print(f"  Vessels near cables: {cable_threat_count}")
    print(f"  Dark vessels near cables: {dark_near_cables}")
    if dark_near_cables > 0:
        print(f"  ⚠️  CRITICAL: Dark vessels detected near submarine cables!")
else:
    print(f"  No immediate threats to cable infrastructure")

print(f"\nRECOMMENDATIONS:")
if detection_report['summary']['high_risk_vessels'] > 0:
    print(f"  🚨 IMMEDIATE: Alert maritime authorities about high-risk vessels")
    print(f"  📡 PRIORITY: Increase monitoring frequency for dark vessels")
if len(vessels_near_cables) > 0:
    print(f"  🔍 MONITOR: Track all vessels near critical infrastructure")
if detection_report['summary']['total_dark_vessels'] > 3:
    print(f"  📊 ANALYZE: High dark vessel count requires pattern analysis")

print(f"  ✅ CONTINUE: Maintain regular surveillance of Arctic waters")
print(f"\n" + "="*60)

## 7. Next Steps

This initial exploration demonstrates the core capabilities of ArcticShadowTracker:

1. **Vessel Detection**: Successfully identified potential dark vessels
2. **Infrastructure Monitoring**: Mapped critical submarine cables
3. **Risk Assessment**: Calculated threat levels for detected vessels
4. **Geographic Analysis**: Visualized vessel distribution and activities

### Recommended next steps:

1. **Real Data Integration**: Connect to live Sentinel-1 and AIS feeds
2. **Model Training**: Use historical data to train the autoencoder
3. **Pattern Analysis**: Implement behavioral pattern recognition
4. **Automated Alerts**: Set up real-time monitoring and alerting
5. **Validation**: Test against known vessel activities

Proceed to `02_autoencoder_training.ipynb` to develop the machine learning models.